# FloodOps SG — RL Flood Prediction


**Pipeline:**
1. Install deps
2. Define the Gym environment (matches live sensor features in the app)
3. Train DQN → `flood_policy.pth`
4. Export → `flood_policy.onnx`
5. Copy `flood_policy.onnx` to `public/` in the repo

____
### 1. Imports

In [9]:
import numpy as np
import gymnasium as gym
from gymnasium import spaces
import onnxruntime as ort   # pip install onnxruntime if needed
import numpy as np
import torch
import torch.nn as nn
import random
import torch.optim as optim
from collections import deque


____
### 2. Gym Environment
- simulates a single Singapore flood zone over up to 60 timesteps
- the agent observes **9 live sensor features** each step and must choose an alert level
- episode ends when a flood fires (any action) or after 60 steps

**State (9 features):**
| Feature | Source in app | Normalisation |
|---------|--------------|---------------|
| `rainfallMm` | data.gov.sg `/rainfall` | ÷ 150 |
| `rainfall_1h` | rolling mean of last 6 readings (~1 hour) | ÷ 150 |
| `rainfall_3h` | rolling mean of last 18 readings (~3 hours) | ÷ 150 |
| `waterLevelPercent` | `zone.waterLevelPercent` | ÷ 100 |
| `waterLevelTrend` | Δwater / 10, clipped −1..1 → rescaled 0..1 | 0 = falling, 1 = rising |
| `trend_rising` | derived from rainfall delta | one-hot |
| `trend_stable` | derived from rainfall delta | one-hot |
| `trend_falling` | derived from rainfall delta | one-hot |
| `official` | signal type === 'official' | 0 / 1 |

**Actions:** 0 = no alert · 1 = watch · 2 = warning · 3 = flash flood alert

**Reward:** correct early escalation = +10, false alarm = −8, missed flood = −20

**Termination:** `step ≥ 60` OR `flood fires` (episode ends on any flood event regardless of action taken)

**Risk formula** (mirrors `zoneRiskScore()` in the TypeScript app, extended with accumulated rainfall):
```
risk = 0.35 × (rain_now / 100)
     + 0.25 × (rain_3h  / 80)    ← accumulated rain weighted heavily
     + 0.25 × (water    / 100)
     + 0.15 × official
```

#### 2.1 FloodEnv Class

In [10]:
class FloodEnv(gym.Env):
    metadata = {"render_modes": []}

    def __init__(self):
        super().__init__()
        self.observation_space = spaces.Box(
            low=np.zeros(9, dtype=np.float32),
            high=np.ones(9, dtype=np.float32),
        )
        self.action_space = spaces.Discrete(4)
        # 0 - No Alert, 1 - watch, 2 - warning, 3 - flash flood alert
        self.max_steps = 60

    def _obs(self):
        # rainfall trend one-hot
        trend_vec = [0.0, 0.0, 0.0]
        trend_vec[["rising", "stable", "falling"].index(self._trend)] = 1.0
        # water level trend: normalised -1..1 -> 0..1
        wl_trend = np.clip((self._water - self._water_prev) / 10.0, -1, 1) * 0.5 + 0.5
        # accumulated rainfall
        rain_1h = np.mean(list(self._rain_history))    / 150.0
        rain_3h = np.mean(list(self._rain_history_3h)) / 150.0
        return np.array([
            self._rain  / 150.0,   # current rainfall
            rain_1h,               # 1-hour accumulated
            rain_3h,               # 3-hour accumulated
            self._water / 100.0,   # water level
            wl_trend,              # water level trend (0=falling, 1=rising)
            *trend_vec,            # rainfall trend one-hot (3 values)
            float(self._official),
        ], dtype=np.float32)       # 9 features total

    def _risk(self):
        rain_3h = np.mean(list(self._rain_history_3h))
        return (0.35 * min(self._rain  / 100.0, 1.0)
              + 0.25 * min(rain_3h     / 80.0,  1.0)
              + 0.25 * min(self._water / 100.0, 1.0)
              + 0.15 * float(self._official))

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self._rain       = self.np_random.uniform(0, 60)
        self._water      = self.np_random.uniform(10, 55)
        self._water_prev = self._water
        self._trend      = self.np_random.choice(["rising", "stable", "falling"])
        self._official   = self.np_random.random() < 0.08
        self._step       = 0
        self._prev_rain  = self._rain
        # 6-step history ~1 hour, 18-step history ~3 hours
        self._rain_history    = deque([self._rain] * 6,  maxlen=6)
        self._rain_history_3h = deque([self._rain] * 18, maxlen=18)
        return self._obs(), {}

    def step(self, action):
        self._step      += 1
        self._water_prev = self._water

        delta = (self.np_random.uniform(3,  12) if self._trend == "rising"  else
                 self.np_random.uniform(-8,   0) if self._trend == "falling" else
                 self.np_random.uniform(-3,   3))
        self._rain  = float(np.clip(self._rain + delta, 0, 150))
        self._water = float(np.clip(
            self._water + (self.np_random.uniform(0, 4) if self._rain > 35
                           else self.np_random.uniform(-1.5, 0.5)), 0, 100))

        self._rain_history.append(self._rain)
        self._rain_history_3h.append(self._rain)

        diff = self._rain - self._prev_rain
        self._trend     = "rising" if diff > 3 else "falling" if diff < -3 else "stable"
        self._prev_rain  = self._rain
        self._official   = self._official or (
            self._rain > 75 and self.np_random.random() < 0.2)

        risk  = self._risk()
        flood = risk > 0.62 and self.np_random.random() < risk
        # removed uncaught_flood early termination — it caused training collapse
        done  = self._step >= self.max_steps

        if flood:
            reward = {3: 10.0, 2: 5.0, 1: 2.0, 0: -20.0}[action]
            done   = True
        elif action == 3 and risk < 0.25: reward = -8.0
        elif action == 2 and risk < 0.15: reward = -4.0
        elif action == 0 and risk < 0.25: reward =  0.5
        else:                              reward =  0.0

        return self._obs(), reward, done, False, {"flood": flood}



In [11]:
#sanity check
env = FloodEnv()
obs, _ = env.reset(seed=42)
print("obs shape:", obs.shape, "| sample obs:", obs.round(3))
obs2, r, done, _, info = env.step(0)
print("step reward:", r, "| done:", done, "| flood:", info["flood"])

obs shape: (9,) | sample obs: [0.31  0.31  0.31  0.297 0.5   0.    1.    0.    0.   ]
step reward: 0.0 | done: False | flood: False


##### 2.11 Explanation of code

```python
self.observation_space = spaces.Box(low=0, high=1, shape=(9,))
self.action_space = spaces.Discrete(4)
self.max_steps = 60
```
- `spaces.Box(low=0, high=1, shape=(9,))` : declares the observation as a **9-element** float array, all values normalised to [0, 1]
- `spaces.Discrete(4)` : declares 4 discrete actions (0 = no alert → 3 = flash flood)
- `self.max_steps = 60` : hard cap — episode terminates after 60 steps if no flood ends it sooner

```python
self._rain_history    = deque([self._rain] * 6,  maxlen=6)
self._rain_history_3h = deque([self._rain] * 18, maxlen=18)
```
- two rolling buffers initialised with the starting rainfall so early predictions aren't noisy
- `maxlen=6` ≈ 1 hour of readings (each step ≈ 10 min); `maxlen=18` ≈ 3 hours

```python
def _obs(self):
    wl_trend = np.clip((self._water - self._water_prev) / 10.0, -1, 1) * 0.5 + 0.5
    rain_1h  = np.mean(list(self._rain_history))    / 150.0
    rain_3h  = np.mean(list(self._rain_history_3h)) / 150.0
    return np.array([rain_now, rain_1h, rain_3h, water, wl_trend, *trend_vec, official])
```
- `wl_trend` : water level change since last step, scaled to [0, 1] (0 = strongly falling, 0.5 = stable, 1 = strongly rising)
- `rain_1h` / `rain_3h` : accumulated rainfall means — the strongest real-world predictor of Singapore flash floods
- trend is one-hot encoded so the network sees three separate binary signals

```python
def _risk(self):
    rain_3h = np.mean(list(self._rain_history_3h))
    return 0.35 * min(rain_now/100, 1) + 0.25 * min(rain_3h/80, 1)
          + 0.25 * min(water/100,   1) + 0.15 * official
```
- `rain_3h` replaces the old social-signal term — accumulated rainfall is a far stronger flood predictor
- flood threshold lowered to `risk > 0.62` to reflect the improved feature set

```python
done = self._step >= self.max_steps
if flood:
    reward = {3: 10.0, 2: 5.0, 1: 2.0, 0: -20.0}[action]
    done   = True
```
- the old `uncaught_flood` early-termination has been **removed** — it caused training collapse where the agent learned to always issue no_alert to avoid seeing the penalty repeatedly
- instead, any flood event ends the episode via `done = True` inside the flood branch
- the agent is rewarded or penalised based on its action at the moment of the flood, then the episode cleanly ends

```python
return self._obs(), reward, done, False, {"flood": flood}
```
- `info["flood"]` : True whenever a flood fires — used to compute detection rate in `evaluate`
- `uncaught_flood` key removed from info (no longer relevant)

##### 2.12 Over-Arching View:
```
reset()  →  randomise rain, water, trend, official  →  init rolling buffers  →  _obs()
                                                                                    │
step(action) ──► evolve dynamics ──► append to rolling buffers ──► _risk() ──► flood?
                                                                                    │
                                                   done=True if step ≥ 60          │
                                                   done=True if flood fires  ◄──────┘
                                                         │
                                                     reward based on action taken
                                                         │
                                                      _obs()  →  return
```
- every episode starts with different initial conditions so the agent generalises across risk levels
- the asymmetric reward (+10 correct escalation / −20 missed flood) pushes the policy to escalate early
- **all** flood events end the episode cleanly — the previous `uncaught_flood` early-termination that caused training collapse has been removed
- rolling 1-hour and 3-hour rainfall buffers give the agent temporal context so it can distinguish a brief spike from a sustained downpour

____
### 3. DQN Model

#### 3.1 DQN Architecture
- learns a Q-function `Q(s, a)` estimating total future reward for each (state, action) pair
- picks the action with the highest Q-value at every step: `action = argmax_a Q(s, a)`
- trained by minimising the Bellman error: `loss = MSE(Q(s,a),  r + γ · max_a' Q_target(s', a'))`
- `γ = 0.95` — near-future rewards weighted almost equally to immediate reward

In [12]:
class DQN(nn.Module):
    """9 inputs → 128 → 128 → 4 Q-values (one per alert level)."""
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(9, 128), nn.ReLU(),
            nn.Linear(128, 128), nn.ReLU(),
            nn.Linear(128, 4),
        )
    def forward(self, x):
        return self.net(x)

##### 3.11 Explanation of code

```python
self.net = nn.Sequential(
    nn.Linear(9, 128), nn.ReLU(),
    nn.Linear(128, 128), nn.ReLU(),
    nn.Linear(128, 4),
)
```
- `nn.Linear(9, 128)` : first hidden layer — maps **9 normalised sensor features** to 128 neurons (1,152 weights + 128 biases = 1,280 params)
- `nn.ReLU()` : non-linear activation — without this, stacking linear layers collapses to a single linear transform regardless of depth
- `nn.Linear(128, 128)` : second hidden layer — learns combinations of the first layer's patterns (16,384 + 128 = 16,512 params)
- `nn.Linear(128, 4)` : output layer — one raw Q-value per alert level (512 + 4 = 516 params); no activation so Q-values are unbounded
- **total: 18,308 trainable parameters**

```python
def forward(self, x):
    return self.net(x)
```
- `x` shape: `(batch, 9)` → output shape: `(batch, 4)`
- during action selection `argmax` is taken over the 4 outputs; during training `.gather(1, actions)` picks the Q-value for the chosen action

____
### 4. Replay Buffer

#### 4.1 Creating ReplayBuffer
- stores up to 10,000 transitions `(obs, action, reward, next_obs, done)` in a circular buffer
- when full, the oldest entry is silently overwritten — no episode tracking needed
- `sample()` returns randomly drawn batches as tensors ready for network training

In [13]:
class ReplayBuffer:
    def __init__(self, capacity=10_000):
        self.capacity = capacity
        self.buffer   = []
        self.position = 0  # write-head for circular overwrite

    def add(self, obs, action, reward, next_obs, done):
        transition = (obs, action, reward, next_obs, done)
        if len(self.buffer) < self.capacity:
            self.buffer.append(transition)
        else:
            self.buffer[self.position] = transition
            self.position = (self.position + 1) % self.capacity

    def sample(self, batch_size):
        indices = np.random.choice(len(self.buffer), batch_size, replace=False)
        batch   = [self.buffer[i] for i in indices]
        obs, actions, rewards, next_obs, dones = zip(*batch)
        return (
            torch.tensor(np.array(obs),      dtype=torch.float32),
            torch.tensor(np.array(actions),  dtype=torch.long),
            torch.tensor(np.array(rewards),  dtype=torch.float32),
            torch.tensor(np.array(next_obs), dtype=torch.float32),
            torch.tensor(np.array(dones),    dtype=torch.float32),
        )
    def __len__(self):
        return len(self.buffer)


##### 4.11 Explanation of code

```python
def add(self, obs, action, reward, next_obs, done):
    if len(self.buffer) < self.capacity:
        self.buffer.append(transition)
    else:
        self.buffer[self.position] = transition
        self.position = (self.position + 1) % self.capacity
```
- `if len(self.buffer) < self.capacity` : while the buffer has room, simply grow it
- once full, `self.buffer[self.position] = transition` : overwrites the oldest entry at the write-head
- `self.position = (self.position + 1) % self.capacity` : advances the write-head by 1, wrapping around when it reaches the end

```python
def sample(self, batch_size):
    indices = np.random.choice(len(self.buffer), batch_size, replace=False)
    batch   = [self.buffer[i] for i in indices]
    obs, actions, rewards, next_obs, dones = zip(*batch)
    return (torch.tensor(...), ...)
```
- `np.random.choice(..., replace=False)` : samples without replacement so no transition appears twice in the same batch
- `zip(*batch)` : transposes list-of-tuples into tuple-of-lists for tensor conversion
- returns 5 tensors — `actions` is `torch.long` for use with `.gather()`; all others are `torch.float32`

____
### 5. Training the DQN Model

#### 5.1 Hyperparameters

In [14]:
EPISODES   = 5_000 #number of episodes training will run for 
BATCH      = 64 # transitions sampled from buffer per update step
LR         = 1e-3 # adam optimiser learning rate
GAMMA      = 0.95 # discount factor, how much the agent values future rewards
EPS_START  = 1.0 # starting epsilon value
EPS_MIN    = 0.01 # minimum epsilon 
EPS_DECAY  = 0.995 #amount epsilon decays by per ep
TARGET_UPD = 500 #episodes between target networks synced
REPLAY_CAP = 10_000 #max transitions stored in replay buffer 

#### 5.2 Training Loop

In [15]:
def train():
    env = FloodEnv()
    policy = DQN()
    target = DQN()
    target.load_state_dict(policy.state_dict())
    target.eval()
    optimizer = optim.Adam(policy.parameters(), lr=LR)
    replay = ReplayBuffer(capacity=REPLAY_CAP)
    epsilon = EPS_START
    rewards_window = deque(maxlen=200)
    for ep in range(1, EPISODES + 1):
        obs, _ = env.reset()
        total  = 0.0
        while True:
            if random.random() < epsilon:
                action = env.action_space.sample()
            else:
                with torch.no_grad():
                    action = policy(torch.tensor(obs).unsqueeze(0)).argmax().item()
            next_obs, reward, done, _, _ = env.step(action)
            replay.add(obs, action, reward, next_obs, float(done))
            obs    = next_obs
            total += reward
            if len(replay) >= BATCH:
                s, a, r, ns, d = replay.sample(BATCH)
                q_pred = policy(s).gather(1, a.unsqueeze(1)).squeeze()
                with torch.no_grad():
                    q_target = r + GAMMA * target(ns).max(1).values * (1 - d)
                loss = nn.functional.mse_loss(q_pred, q_target)
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
            if done:
                break
        epsilon = max(EPS_MIN, epsilon * EPS_DECAY)
        rewards_window.append(total)

        if ep % TARGET_UPD == 0:
            target.load_state_dict(policy.state_dict())
        if ep % 500 == 0:
            avg = sum(rewards_window) / len(rewards_window)
            print(f"Episode {ep:>5}/{EPISODES}  avg_reward={avg:+.2f}  eps={epsilon:.3f}")

    print("\nTraining complete.")
    return policy


##### 5.21 Explanation of code

```python
replay.add(obs, action, reward, next_obs, float(done))
```
- stores each transition into the `ReplayBuffer` after every environment step

```python
if len(replay) >= BATCH:
    s, a, r, ns, d = replay.sample(BATCH)
```
- waits until the buffer has at least `BATCH=64` transitions before starting training
- `replay.sample()` returns 5 tensors already — no manual `torch.tensor(...)` conversion needed

```python
q_pred = policy(s).gather(1, a.unsqueeze(1)).squeeze()
```
- `policy(s)` : forward pass gives Q-values for all 4 actions, shape `(64, 4)`
- `.gather(1, a.unsqueeze(1))` : selects the Q-value for the action that was actually taken
- `.squeeze()` : collapses to shape `(64,)` to match `q_target`

```python
q_target = r + GAMMA * target(ns).max(1).values * (1 - d)
```
- `target(ns).max(1).values` : best Q-value from the next state according to the frozen target network
- `* (1 - d)` : zeroes out future value when the episode ended (`done=1`), so terminal states only count immediate reward
- `torch.no_grad()` : target values are fixed labels — no gradients needed

```python
if ep % TARGET_UPD == 0:
    target.load_state_dict(policy.state_dict())
```
- hard-copies the policy weights into the target network every 500 episodes
- prevents the Bellman target from chasing itself (instability) by keeping the target frozen between syncs

#### 5.3 Evaluating the Model
- runs `n_episodes` episodes with a fully greedy policy (no exploration, ε = 0)
- tracks reward, flood detection rate, false alarm rate, and action distribution across episodes

In [16]:
def evaluate(policy, n_episodes=200, seed=0):
    LABELS = ["no_alert", "watch", "warning", "flash_flood"]
    env    = FloodEnv()
    policy.eval()

    total_rewards = []
    action_counts = [0, 0, 0, 0]
    floods_total  = 0
    floods_caught = 0  # flood occurred and agent issued action >= 1
    false_alarms  = 0  # action == 3 when risk < 0.3

    for ep in range(n_episodes):
        obs, _    = env.reset(seed=seed + ep)
        ep_reward = 0.0

        while True:
            with torch.no_grad():
                action = policy(torch.tensor(obs).unsqueeze(0)).argmax().item()

            obs, reward, done, _, info = env.step(action)
            ep_reward += reward
            action_counts[action] += 1

            if info["flood"]:
                floods_total += 1
                if action >= 1:
                    floods_caught += 1
            if action == 3 and env._risk() < 0.3:
                false_alarms += 1

            if done:
                break

        total_rewards.append(ep_reward)

    avg_reward     = sum(total_rewards) / n_episodes
    detection_rate = floods_caught / floods_total if floods_total > 0 else float("nan")
    total_steps    = sum(action_counts)

    print(f"Evaluation over {n_episodes} episodes")
    print(f"  Avg reward      : {avg_reward:+.2f}")
    print(f"  Flood events    : {floods_total}")
    print(f"  Detection rate  : {detection_rate:.1%}  ({floods_caught}/{floods_total} floods with action >= 1)")
    print(f"  False alarms    : {false_alarms}  (action=3 when risk < 0.3)")
    print(f"  Action distribution:")
    for label, count in zip(LABELS, action_counts):
        print(f"    {label:<15}  {count:>5}  ({count / total_steps:.1%})")

    return {
        "avg_reward":     avg_reward,
        "detection_rate": detection_rate,
        "false_alarms":   false_alarms,
        "action_counts":  action_counts,
        "rewards":        total_rewards,
    }



##### 5.31 Explanation of code

```python
policy.eval()
```
- switches off dropout/batchnorm training behaviour and disables gradient tracking — required before any inference pass

```python
with torch.no_grad():
    action = policy(torch.tensor(obs).unsqueeze(0)).argmax().item()
```
- `torch.no_grad()` : no gradients computed — evaluation is purely forward passes
- `.unsqueeze(0)` : adds a batch dimension so the input shape is `(1, 9)` as the network expects
- `.argmax().item()` : picks the action with the highest Q-value as a plain Python int

```python
if info["flood"]:
    floods_total += 1
    if action >= 1:
        floods_caught += 1
```
- a flood is "caught" if the agent issued at least a watch (action ≥ 1) at the moment the flood fired
- `floods_caught / floods_total` gives the **detection rate** — the primary safety metric

```python
if action == 3 and env._risk() < 0.3:
    false_alarms += 1
```
- counts steps where the agent issued a flash flood alert under genuinely low-risk conditions
- high false alarm counts indicate the policy is over-triggering

```python
return { "avg_reward": ..., "detection_rate": ..., ... }
```
- returns a dict so results can be compared across checkpoints or used for plotting

#### 5.4 Training and evaluating the model

In [ ]:
policy = train()

In [ ]:
results = evaluate(policy)

____
### 6. Storing the DQN as a Class

#### 6.1 DQNAgent Class
- bundles all state, networks, and logic into one object — mirrors the `TD3Agent` pattern
- `act` : ε-greedy action selection
- `update` : one Bellman step (Double DQN) + hard target-network sync every `target_update` gradient steps
- `train(render=False)` : full training loop, returns `self` so calls can be chained
- `evaluate(render=False)` : greedy evaluation reporting reward, detection rate and false alarms
- `play(render=False)` : runs one episode and returns a step-by-step result dict
- `save(render=False)` : saves policy weights to a `.pth` checkpoint
- `load(render=False)` : classmethod — creates a new agent and loads weights from a `.pth` file

All methods accept `render=False`; pass `render=True` to enable printed output.

In [18]:
class DQNAgent:
    def __init__(self, env=None, gamma=0.95, eps_start=1.0, eps_min=0.01,
                 eps_decay=0.999, batch_size=64, buffer_capacity=10_000,
                 lr=1e-3, target_update=500):
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.env = env if env is not None else FloodEnv()
        self.policy = DQN().to(self.device)
        self.target = DQN().to(self.device)
        self.target.load_state_dict(self.policy.state_dict())
        self.target.eval()
        for param in self.target.parameters():
            param.requires_grad = False
        self.optimizer     = optim.Adam(self.policy.parameters(), lr=lr)
        self.replay_buffer = ReplayBuffer(capacity=buffer_capacity)
        self.gamma         = gamma
        self.eps           = eps_start
        self.eps_min       = eps_min
        self.eps_decay     = eps_decay
        self.batch_size    = batch_size
        self.target_update = target_update
        self.total_updates = 0

    def act(self, obs, explore=True):
        if explore and random.random() < self.eps:
            return self.env.action_space.sample()
        obs_t = torch.tensor(obs, dtype=torch.float32, device=self.device).unsqueeze(0)
        self.policy.eval()
        with torch.no_grad():
            action = self.policy(obs_t).argmax().item()
        self.policy.train()
        return action

    def update(self):
        if len(self.replay_buffer) < self.batch_size:
            return
        s, a, r, ns, d = self.replay_buffer.sample(self.batch_size)
        s, a, r, ns, d = s.to(self.device), a.to(self.device), r.to(self.device), ns.to(self.device), d.to(self.device)
        q_pred = self.policy(s).gather(1, a.unsqueeze(1)).squeeze()
        with torch.no_grad():
            best_actions = self.policy(ns).argmax(1, keepdim=True)
            q_target = r + self.gamma * self.target(ns).gather(1, best_actions).squeeze() * (1 - d)
        loss = nn.functional.mse_loss(q_pred, q_target)
        self.optimizer.zero_grad()
        loss.backward()
        self.optimizer.step()
        self.total_updates += 1
        if self.total_updates % self.target_update == 0:
            self.target.load_state_dict(self.policy.state_dict())

    def train(self, episodes=5_000, log_every=500, render=False):
        rewards_window = deque(maxlen=200)
        for ep in range(1, episodes + 1):
            obs, _ = self.env.reset()
            total  = 0.0
            while True:
                action = self.act(obs, explore=True)
                next_obs, reward, done, _, _ = self.env.step(action)
                self.replay_buffer.add(obs, action, reward, next_obs, float(done))
                obs    = next_obs
                total += reward
                self.update()
                if done:
                    break
            self.eps = max(self.eps_min, self.eps * self.eps_decay)
            rewards_window.append(total)
            if render and ep % log_every == 0:
                avg = sum(rewards_window) / len(rewards_window)
                print(f"Episode {ep:>5}/{episodes}  avg_reward={avg:+.2f}  eps={self.eps:.3f}")
        print("\nTraining complete.")
        return self

    def evaluate(self, n_episodes=31, seed=0, render=False):
        LABELS = ["no_alert", "watch", "warning", "flash_flood"]
        env = FloodEnv()
        self.policy.eval()
        total_rewards = []
        action_counts = [0, 0, 0, 0]
        floods_total  = 0
        floods_caught = 0
        false_alarms  = 0
        for ep in range(n_episodes):
            obs, _    = env.reset(seed=seed + ep)
            ep_reward = 0.0
            while True:
                action = self.act(obs, explore=False)
                obs, reward, done, _, info = env.step(action)
                ep_reward += reward
                action_counts[action] += 1
                if info["flood"]:
                    floods_total += 1
                    if action >= 1:
                        floods_caught += 1
                if action == 3 and env._risk() < 0.3:
                    false_alarms += 1
                if done:
                    break
            total_rewards.append(ep_reward)
        self.policy.train()
        avg_reward     = sum(total_rewards) / n_episodes
        detection_rate = floods_caught / floods_total if floods_total > 0 else float("nan")
        total_steps    = sum(action_counts)
        if render:
            print(f"Evaluation over {n_episodes} episodes")
            print(f"  Avg reward      : {avg_reward:+.2f}")
            print(f"  Flood events    : {floods_total}")
            print(f"  Detection rate  : {detection_rate:.1%}  ({floods_caught}/{floods_total} floods with action >= 1)")
            print(f"  False alarms    : {false_alarms}  (action=3 when risk < 0.3)")
            print(f"  Action distribution:")
            for label, count in zip(LABELS, action_counts):
                print(f"  {label:<15}  {count:>5}  ({count / total_steps:.1%})")
        return {
            "avg_reward":     avg_reward,
            "detection_rate": detection_rate,
            "false_alarms":   false_alarms,
            "action_counts":  action_counts,
            "rewards":        total_rewards,
        }

    def play(self, seed=None, render=False): #simulates for 1 day 
        LABELS = ["no_alert", "watch", "warning", "flash_flood"]
        TRENDS = ["rising", "stable", "falling"]
        env = FloodEnv()
        obs, _ = env.reset(seed=seed)
        self.policy.eval()
        total   = 0.0
        step    = 0
        actions = []
        if render:
            print(f"{'Step':>4}  {'Rain':>6}  {'Water':>6}  {'Trend':<8}  {'Risk':>5}  {'Action':<15}  {'Reward':>7}  Note")
            print("-" * 72)
        while True:
            step  += 1
            action = self.act(obs, explore=False)
            obs, reward, done, _, info = env.step(action)
            total += reward
            actions.append(action)
            if render:
                trend_idx = int(np.argmax(obs[2:5]))
                note = "*** FLOOD ***" if info["flood"] else ""
                print(
                    f"{step:>4}  "
                    f"{env._rain:>6.1f}  "
                    f"{env._water:>6.1f}  "
                    f"{TRENDS[trend_idx]:<8}  "
                    f"{env._risk():>5.2f}  "
                    f"{LABELS[action]:<15}  "
                    f"{reward:>+7.1f}  "
                    f"{note}"
                )
            if done:
                break
        self.policy.train()
        if render:
            print("-" * 72)
            print(f"Episode ended  |  steps: {step}  |  total reward: {total:+.1f}")
        return {"total_reward": total, "steps": step, "actions": actions}

    def save(self, path="flood_policy.pth", render=False):
        torch.save(self.policy.state_dict(), path)
        if render:
            print(f"Saved {path}")
        return self

    @classmethod
    def load(cls, path="flood_policy.pth", render=False, **kwargs):
        agent = cls(eps_start=0.0, **kwargs)
        state_dict = torch.load(path, map_location=agent.device)
        agent.policy.load_state_dict(state_dict)
        agent.target.load_state_dict(state_dict)
        agent.policy.eval()
        if render:
            print(f"Loaded {path}")
        return agent

##### 6.11 Explanation of code

```python
self.policy = DQN().to(self.device)
self.target = DQN().to(self.device)
self.target.load_state_dict(self.policy.state_dict())
for param in self.target.parameters():
    param.requires_grad = False
```
- creates two identical networks; `target` is a frozen copy that provides stable Bellman labels
- `requires_grad = False` ensures the target never accumulates gradients — it is only ever updated via hard copy

```python
def act(self, obs, explore=True):
    if explore and random.random() < self.eps:
        return self.env.action_space.sample()
    action = self.policy(obs_t).argmax().item()
```
- `explore=True` during training (ε-greedy), `explore=False` during evaluation (fully greedy)
- `self.policy.eval()` / `.train()` toggles batchnorm/dropout behaviour around each inference call

```python
def update(self):
    best_actions = self.policy(ns).argmax(1, keepdim=True)
    q_target = r + self.gamma * self.target(ns).gather(1, best_actions).squeeze() * (1 - d)
    if self.total_updates % self.target_update == 0:
        self.target.load_state_dict(self.policy.state_dict())
```
- Double DQN: `self.policy` picks the best next action, `self.target` evaluates it — reduces Q-value overestimation vs vanilla DQN
- hard-syncs the target every `target_update` gradient steps (not per episode)

```python
def train(self, episodes=5_000, log_every=500, render=False):
    if render and ep % log_every == 0: print(...)
    if render: print("Training complete.")
    return self
```
- `render=False` by default — silent training for use inside loops or pipelines
- `return self` allows chaining: `agent = DQNAgent().train()`

```python
def evaluate(self, n_episodes=31, seed=0, render=False):
    self.policy.eval()
    ...
    self.policy.train()
    if render: print(metrics)
    return { "avg_reward": ..., "detection_rate": ..., ... }
```
- always returns the metrics dict regardless of `render`
- `self.policy.train()` restores training mode so `agent.train()` can be called after evaluate without side effects

```python
def play(self, seed=None, render=False):
```
- runs one greedy episode; `render=True` prints the step-by-step table
- returns `{"total_reward", "steps", "actions"}` — useful for scripted simulations

```python
def save(self, path="flood_policy.pth", render=False):
    torch.save(self.policy.state_dict(), path)
```
- saves only the policy weights (not optimizer or buffer) — sufficient for inference and fine-tuning

```python
@classmethod
def load(cls, path="flood_policy.pth", render=False, **kwargs):
    agent = cls(eps_start=0.0, **kwargs)
    agent.policy.load_state_dict(torch.load(path, map_location=agent.device))
    agent.target.load_state_dict(state_dict)
```
- `@classmethod` — called as `DQNAgent.load("flood_policy.pth")` without needing an existing instance
- `eps_start=0.0` makes the loaded agent act greedily immediately; set `agent.eps = 0.1` before `agent.train()` to resume training
- `**kwargs` forwards any hyperparameter overrides (e.g. `gamma=0.99`) to `__init__`

#### 6.2 Train and Evaluate

In [19]:
agent = DQNAgent().train(episodes = 20_000)



Training complete.


In [20]:
eval = agent.evaluate(render = True)

Evaluation over 31 episodes
  Avg reward      : +11.66
  Flood events    : 15
  Detection rate  : 100.0%  (15/15 floods with action >= 1)
  False alarms    : 10  (action=3 when risk < 0.3)
  Action distribution:
  no_alert           676  (59.7%)
  watch              114  (10.1%)
  warning            207  (18.3%)
  flash_flood        135  (11.9%)


In [22]:
result = agent.play()

#### 6.3 Save PyTorch Checkpoint

In [23]:
agent.save()

#### 6.4 Load from checkpoint

In [25]:
agent = DQNAgent.load("flood_policy.pth")
res = agent.play(seed=42)

C:\Users\xuan2\AppData\Local\Temp\ipykernel_15108\1973659009.py:167: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load(path, map_location=agent.device)


### 7. Performance of model for an episode 

In [26]:
def simulation(n = 1):
    final = []
    model = DQNAgent.load("flood_policy.pth")
    for i in range(0, n):
        results = model.play(seed = i)
        final.append(results)
    return final

def print_sim_res(res):
    for n, day in enumerate(res, 1):
        print(f"day {n}: {day}")
    rewards = [d["total_reward"] for d in res]
    steps   = [d["steps"]        for d in res]
    n_days  = len(res)
    print(f"\n--- Average over {n_days} episodes ---")
    print(f"  Avg total reward : {sum(rewards) / n_days:+.2f}")
    print(f"  Avg steps        : {sum(steps)   / n_days:.1f}")
    print(f"  Min reward       : {min(rewards):+.2f}")
    print(f"  Max reward       : {max(rewards):+.2f}")

In [27]:
month = simulation(31)

C:\Users\xuan2\AppData\Local\Temp\ipykernel_15108\1973659009.py:167: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load(path, map_location=agent.device)


In [28]:
print_sim_res(month)

day 1: {'total_reward': 10.0, 'steps': 5, 'actions': [3, 3, 3, 3, 3]}
day 2: {'total_reward': 5.0, 'steps': 6, 'actions': [1, 1, 1, 1, 0, 2]}
day 3: {'total_reward': 30.0, 'steps': 60, 'actions': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]}
day 4: {'total_reward': 12.5, 'steps': 18, 'actions': [0, 0, 0, 0, 0, 0, 1, 1, 2, 3, 0, 3, 3, 3, 3, 3, 3, 3]}
day 5: {'total_reward': 10.0, 'steps': 25, 'actions': [0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 2, 2, 3, 3, 3, 3, 3, 3, 3]}
day 6: {'total_reward': 0.0, 'steps': 60, 'actions': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 0, 0, 0, 0, 0, 2, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]}
day 7: {'total_reward': 15.0, 'steps': 60, 'actions': [3, 2, 2, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0

___
### 7. Exporting model weights to json format for use in FloodOpsSg


In [29]:
import json

# agent.policy is the nn.Module — agent itself is not an nn.Module
weights = {name: param.detach().numpy().tolist()
           for name, param in agent.policy.named_parameters()}

with open("flood_policy_weights.json", "w") as f:
    json.dump(weights, f)

print("Keys:", list(weights.keys()))
print("Done → flood_policy_weights.json")

Keys: ['net.0.weight', 'net.0.bias', 'net.2.weight', 'net.2.bias', 'net.4.weight', 'net.4.bias']
Done → flood_policy_weights.json
